# ML10 · Autoencoder 自編碼器

用神經網路把高維資料壓成精簡的潛在表示（latent representation），再還原回來，藉「重建得有多像」學會資料的重要結構。

## 學習目標
- 理解自編碼器（autoencoder）的整體結構：編碼器（encoder）與解碼器（decoder）的串接。
- 認識瓶頸（bottleneck）與潛在表示（latent representation）的角色，知道壓縮為何能逼出重點。
- 理解重建損失（reconstruction loss）如何驅動訓練，讓輸出逼近輸入。
- 能用自造的多維特徵資料，建立並訓練一個最小自編碼器。
- 會視覺化重建誤差（reconstruction error），判讀哪些樣本被還原得好或不好。
- 理解潛在維度（latent dimension）大小對壓縮率與重建品質的取捨。

In [ ]:
# 概念：環境與字型 setup（之後會畫損失曲線、長條圖與散布圖）
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# 讓中文標題與標籤能正常顯示（依系統挑一個常見字型；找不到就用預設）
matplotlib.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "PingFang TC", "Noto Sans CJK TC", "SimHei", "Arial Unicode MS"]
matplotlib.rcParams["axes.unicode_minus"] = False   # 負號用一般減號顯示，避免變成方框

print("numpy 版本:", np.__version__)

## 為什麼要壓縮：從監督到自監督

前面的分類模型需要人工標籤當答案；自編碼器改用自監督學習（self-supervised learning），讓「資料自己當答案」，目標是把輸入原樣還原回來。

為什麼有用：真實資料常有冗餘，欄位彼此相關（例如樓高、樓層數、總面積會一起變動）。壓縮（降維 dimensionality reduction）迫使模型只保留重要資訊，這正是表示學習（representation learning）的起點。

何時用：沒有標籤、想找資料內在結構、想用少量數字概括一筆資料時。

In [ ]:
# 概念：自造一批帶冗餘欄位的多維特徵，觀察欄位高度相關（彼此可互相推算）就是「可被壓縮」的訊號
import numpy as np

rng = np.random.default_rng(0)   # 固定亂數種子，讓每次結果一致方便對照

# 造 200 棟樓房，先生出一個潛在的「規模因子」，多數欄位都由它衍生（刻意製造冗餘）
n = 200
scale = rng.uniform(1.0, 5.0, size=n)            # 規模因子：越大代表樓越大
floors = np.round(scale * 4 + rng.normal(0, 1, n))      # 樓層數約與規模成正比
height = floors * 3.0 + rng.normal(0, 1.5, n)           # 樓高約等於樓層數乘層高
floor_area = scale * 80 + rng.normal(0, 10, n)          # 單層面積也跟著規模走
total_area = floor_area * floors                        # 總樓地板幾乎可由前兩欄算出

features = np.column_stack([floors, height, floor_area, total_area])
print("特徵 shape (樣本數, 特徵數):", features.shape)

# 相關係數越接近 1 代表欄位越冗餘；高相關就是壓縮空間的來源
corr = np.corrcoef(features, rowvar=False)   # rowvar=False：每欄當一個變數
print("欄位間相關係數矩陣（越接近 1 越冗餘）:")
print(np.round(corr, 2))

## 自編碼器結構：編碼器與解碼器

自編碼器是兩段網路的串接：編碼器（encoder）把輸入一路往下壓到很窄的中間層，解碼器（decoder）再把它展開回原本的尺寸。整體像「漏斗再展開」。

關鍵：輸入與輸出的維度必須相同，因為訓練目標是「輸出要等於輸入」。前向傳遞（forward pass）就是資料從輸入依序穿過每一層、算到輸出的過程。

常見是對稱結構（symmetric architecture），例如 8 → 4 → 2 → 4 → 8，左右維度鏡像對齊。

In [ ]:
# 概念：用 list 描述各層維度，模擬一筆資料做前向傳遞，觀察「形狀先變窄再變回來」
import numpy as np

# 各層輸出維度：左半是編碼器（越來越窄），中間是瓶頸，右半是解碼器（鏡像展開）
layer_dims = [8, 4, 2, 4, 8]

rng = np.random.default_rng(1)
x = rng.normal(size=8)            # 造一筆 8 維輸入當示範

# 為每一層相鄰維度造一個隨機權重矩陣，模擬前向傳遞時形狀如何被改變
activation = x
print("輸入形狀:", activation.shape)
for in_dim, out_dim in zip(layer_dims[:-1], layer_dims[1:]):
    W = rng.normal(size=(in_dim, out_dim)) * 0.3   # 權重把 in_dim 維映射到 out_dim 維
    activation = np.tanh(activation @ W)           # @ 是矩陣乘法；tanh 是常見的非線性活化函數
    print("經過一層 ->", in_dim, "轉", out_dim, " 目前形狀:", activation.shape)

# 注意：輸入是 8 維、最後輸出也是 8 維，輸入輸出同尺寸才有「重建」可言
print("最後輸出形狀:", activation.shape, "（與輸入相同）")

## 瓶頸與潛在表示

整個網路最窄的那一層就是瓶頸（bottleneck）；通過瓶頸後得到的向量就是潛在表示（latent representation），它是整筆資料的精簡座標。

為什麼瓶頸逼出重點：資訊必須擠過這個窄口，模型只能保留最能幫助還原的特徵，丟掉冗餘，這就是資訊壓縮（information compression）。潛在維度（latent dimension）就是瓶頸的寬度。

直覺：瓶頸越窄壓縮越狠，但壓太狠會還原失真。下面用最簡單的線性壓縮（取資料主要方向）來感受這個取捨。

In [ ]:
# 概念：把瓶頸維度設為 1、2、3，用線性投影壓縮再還原，比較還原誤差，感受「壓太狠會失真」
import numpy as np

rng = np.random.default_rng(2)
# 造 150 筆 4 維資料，但其實只由 2 個獨立方向構成（內在維度約為 2）
base = rng.normal(size=(150, 2))
mix = rng.normal(size=(2, 4))            # 把 2 維混成 4 維（製造冗餘）
data = base @ mix
data = (data - data.mean(axis=0)) / data.std(axis=0)   # 先標準化，讓各欄尺度一致

# 技巧：用 SVD 取資料的主要方向當「線性瓶頸」，是自編碼器壓縮的線性版直覺
U, S, Vt = np.linalg.svd(data, full_matrices=False)

for latent_dim in [1, 2, 3]:
    directions = Vt[:latent_dim]                 # 取前 latent_dim 個主要方向
    latent = data @ directions.T                 # encode：投影到瓶頸維度
    reconstructed = latent @ directions          # decode：用同方向展開回 4 維
    mse = np.mean((data - reconstructed) ** 2)   # 重建誤差（均方誤差）
    print(f"瓶頸維度 = {latent_dim}  ->  平均重建誤差 = {mse:.4f}")

# 注意：因為真實內在維度約為 2，瓶頸到 2 之後誤差就大幅下降，再加維度收益有限

## 重建損失與訓練流程

訓練目標只有一句話：讓輸出逼近輸入。衡量「逼近程度」的就是重建損失（reconstruction loss），最常用均方誤差（mean squared error, MSE）：把每個元素的誤差平方後取平均。

怎麼變好：用梯度下降（gradient descent），每個 epoch 算出損失對權重的梯度，往讓損失下降的方向微調權重，一步步把損失降下來。一個訓練迴圈（training loop）就是反覆做「前向算損失 → 反向算梯度 → 更新權重」。

公式：MSE = 平均((輸入 − 輸出)^2)。

In [ ]:
# 概念：手刻一個最小線性自編碼器，用 MSE 當損失、用梯度下降訓練數十個 epoch，收集並畫出損失曲線
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(3)
# 造 300 筆 6 維資料，內在維度約 2（方便壓到很窄仍能還原）
base = rng.normal(size=(300, 2))
X = base @ rng.normal(size=(2, 6))
X = (X - X.mean(axis=0)) / X.std(axis=0)   # 標準化是穩定訓練的常見前處理

input_dim, latent_dim = 6, 2
# 編碼器與解碼器各用一個權重矩陣（線性自編碼器，無偏置與非線性，聚焦訓練直覺）
W_enc = rng.normal(size=(input_dim, latent_dim)) * 0.1
W_dec = rng.normal(size=(latent_dim, input_dim)) * 0.1

lr = 0.05            # 學習率：每步往下走的幅度
losses = []          # 收集每個 epoch 的損失，之後畫曲線
for epoch in range(80):
    latent = X @ W_enc            # 前向：編碼
    recon = latent @ W_dec        # 前向：解碼
    error = recon - X             # 重建誤差（輸出減輸入）
    loss = np.mean(error ** 2)    # MSE：誤差平方的平均
    losses.append(loss)

    # 反向：用鏈鎖法則算損失對兩個權重的梯度（除以樣本數做平均）
    grad_dec = latent.T @ error / X.shape[0]
    grad_enc = X.T @ (error @ W_dec.T) / X.shape[0]
    W_dec -= lr * grad_dec        # 往梯度反方向更新，讓損失下降
    W_enc -= lr * grad_enc

print(f"第一個 epoch 損失: {losses[0]:.4f}   最後一個 epoch 損失: {losses[-1]:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("重建損失 MSE")
plt.title("訓練過程中重建損失逐步下降")
plt.show()

## 用自造特徵示範壓縮與還原

訓練好的自編碼器可以拆成兩段分開用：先用編碼器做 encode 得到潛在向量，需要時再用解碼器 decode 還原。實用價值是「平常只存小向量，要用時再展開」。

壓縮率（compression ratio）= 原始維度 / 潛在維度，數字越大代表存得越省。先做標準化（normalization）讓各欄尺度一致，訓練才穩定。

下面接續上一節訓練好的權重，取幾筆樣本並排比較原始與還原數值。

In [ ]:
# 概念：把模型拆成 encode / decode 兩段使用，並計算壓縮率與還原相似度
import numpy as np

rng = np.random.default_rng(3)
# 重現上一節同一份資料與訓練（自足，不依賴上一個 cell 的變數）
base = rng.normal(size=(300, 2))
X = base @ rng.normal(size=(2, 6))
X = (X - X.mean(axis=0)) / X.std(axis=0)

input_dim, latent_dim = 6, 2
W_enc = rng.normal(size=(input_dim, latent_dim)) * 0.1
W_dec = rng.normal(size=(latent_dim, input_dim)) * 0.1
lr = 0.05
for epoch in range(80):
    latent = X @ W_enc
    recon = latent @ W_dec
    error = recon - X
    grad_dec = latent.T @ error / X.shape[0]
    grad_enc = X.T @ (error @ W_dec.T) / X.shape[0]
    W_dec -= lr * grad_dec
    W_enc -= lr * grad_enc

def encode(batch):
    return batch @ W_enc          # 只走編碼器：高維 -> 潛在向量

def decode(codes):
    return codes @ W_dec          # 只走解碼器：潛在向量 -> 還原高維

sample = X[:3]                    # 取前 3 筆當示範
codes = encode(sample)            # 存下來的就是這個小向量
restored = decode(codes)          # 需要時再展開

compression_ratio = input_dim / latent_dim
print("壓縮率 (原始維度 / 潛在維度):", compression_ratio)
for i in range(3):
    print(f"樣本 {i}  潛在向量:", np.round(codes[i], 3))
    print("   原始:", np.round(sample[i], 2))
    print("   還原:", np.round(restored[i], 2))

## 視覺化重建誤差與潛在空間

對每一筆資料算出它的重建誤差（reconstruction error，自己這筆的 MSE），畫成長條圖就能看出誤差分布（error distribution）。誤差特別大的樣本往往是「不尋常」的，這就是異常偵測的直覺（anomaly intuition）。

把每筆的 2 維潛在向量畫成散布圖（latent scatter），相似的樣本通常聚在一起、不同類別會分開。

下面對全部樣本算逐筆誤差畫長條圖，再把潛在向量畫成依類別上色的散布圖。

In [ ]:
# 概念：算每筆重建誤差畫長條圖，並把 2 維潛在向量畫成依類別上色的散布圖
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(4)
# 造兩群樓房：群 0 規模較小、群 1 規模較大，各 80 筆（潛在類別）
group0 = rng.normal(loc=[-1.5, -1.0], scale=0.4, size=(80, 2))
group1 = rng.normal(loc=[1.5, 1.0], scale=0.4, size=(80, 2))
base = np.vstack([group0, group1])
labels = np.array([0] * 80 + [1] * 80)        # 類別只用來上色，訓練時不使用（自監督）
X = base @ rng.normal(size=(2, 6))
X = (X - X.mean(axis=0)) / X.std(axis=0)

# 訓練同款線性自編碼器
input_dim, latent_dim = 6, 2
W_enc = rng.normal(size=(input_dim, latent_dim)) * 0.1
W_dec = rng.normal(size=(latent_dim, input_dim)) * 0.1
for epoch in range(120):
    latent = X @ W_enc
    recon = latent @ W_dec
    error = recon - X
    W_dec -= 0.05 * (latent.T @ error / X.shape[0])
    W_enc -= 0.05 * (X.T @ (error @ W_dec.T) / X.shape[0])

latent = X @ W_enc
recon = latent @ W_dec
per_sample_error = np.mean((recon - X) ** 2, axis=1)   # axis=1：每一筆自己的平均誤差

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(range(len(per_sample_error)), per_sample_error)
axes[0].set_xlabel("樣本索引")
axes[0].set_ylabel("重建誤差")
axes[0].set_title("每筆樣本的重建誤差分布")

scatter = axes[1].scatter(latent[:, 0], latent[:, 1], c=labels, cmap="coolwarm", s=20)
axes[1].set_xlabel("潛在維度 1")
axes[1].set_ylabel("潛在維度 2")
axes[1].set_title("潛在空間散布圖（依類別上色）")
plt.tight_layout()
plt.show()

print("平均重建誤差:", round(per_sample_error.mean(), 4))

## 練習

以下三題由淺入深，皆為複合型但簡短。資料請自己用 numpy 造（仿真即可），完成後對照「驗收」確認輸出。

In [ ]:
# TODO 1 ·（簡單）樓房特徵的最小壓縮（整合：編碼器/解碼器 + 重建損失）
#   情境：自造一批樓房資料，欄位含樓高、樓層數、單層面積等彼此相關的多維特徵。
#   要求：
#     1. 用 numpy 自造資料並逐欄標準化，搭一個對稱的小型自編碼器（如 6 -> 2 -> 6）。
#     2. 以均方誤差為重建損失訓練數十個 epoch，記錄並畫出損失曲線。
#   驗收：應該看到損失隨 epoch 明顯下降，且部分樣本的還原值接近原始值。
# TODO: 在這裡完成


In [ ]:
# TODO 2 ·（中間）潛在維度的取捨實驗（整合：瓶頸 + 潛在表示 + 重建誤差 + 壓縮率）
#   情境：用自造的社區地塊資料（面積、容積率、綠地比例、退縮距離等多欄位）做壓縮分析。
#   要求：
#     1. 將瓶頸維度分別設為 1、2、3 各訓練一次。
#     2. 對每種設定計算平均重建誤差與壓縮率（原始維度 / 潛在維度）。
#     3. 把「潛在維度 對 重建誤差」畫成折線圖。
#   驗收：應看到潛在維度越大、重建誤差越小，並能說出一個「夠用又省」的維度選擇。
# TODO: 在這裡完成


In [ ]:
# TODO 3 ·（稍難）用重建誤差找異常街廓（整合：encode/decode + 重建誤差 + 潛在散布 + 異常直覺）
#   情境：自造一批「正常」街廓特徵訓練模型，另外混入少量刻意偏離的異常街廓（如異常高的容積率）。
#   要求：
#     1. 只用正常資料訓練自編碼器，再對全部資料（含異常）做 encode/decode。
#     2. 計算每筆重建誤差，設一個門檻把高誤差樣本標為可疑（例如取正常誤差的某個百分位）。
#     3. 把 2 維潛在向量畫成散布圖，標出被判為異常的點。
#   驗收：應看到刻意混入的異常街廓多半落在高重建誤差區、並在潛在散布圖中偏離主群；
#         並能討論門檻怎麼定較合理。
# TODO: 在這裡完成


## 小結

- 自編碼器是自監督學習：不用標籤，讓資料自己當答案，目標是把輸入原樣還原回來。
- 結構是編碼器與解碼器的串接（常為對稱），輸入與輸出維度相同才有「重建」可言。
- 最窄的瓶頸層產生潛在表示；壓縮迫使模型丟掉冗餘、留下本質，潛在維度就是瓶頸寬度。
- 訓練用重建損失（多用 MSE）驅動，靠梯度下降在訓練迴圈裡一步步把損失降下來。
- 模型可拆成 encode / decode 兩段使用：平常存小向量、需要時再展開，壓縮率為原始維度除以潛在維度。
- 逐筆重建誤差能找出「不尋常」的樣本（異常直覺）；把潛在向量畫成散布圖可看出相似樣本是否聚在一起。
- 潛在維度的選擇是取捨：太窄會失真、太寬不夠省，挑「夠用又省」的維度最實用。